<h1> CNN-Based Classification Model Tutorial </h1>
<h5> <p style="font-size:90% ; font-family:arial"> (1) options.py </p> </h5> 
<h5> <p style="font-size:120% ; line-height:50% ; color:blue ; font-weight:bold">  (2) pipeline.py </p> </h5> 
<h5> <p style="font-size:90% ; line-height:50%">  (3) networks.py </p> </h5> 
<h5> <p style="font-size:90% ; line-height:50%">  (4) train.py </p> </h5> 

In [ ]:
import os
import sys

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    sys.path.append("/content/drive/MyDrive/CBNU/Classification")
    print(os.getcwd())
    os.chdir("/content/drive/MyDrive/CBNU/Classification")
    print(os.getcwd())
except ModuleNotFoundError:
    print("Not in colab, continue")

위 내용은 앞에서 다뤘습니다. <br>

In [ ]:
from options import TrainOptions
opt = TrainOptions().parse()

TrainOptions를 불러와 선언합니다.

In [ ]:
from pipeline import CustomDataset

pipeline.py에서 정의한 CustomDataset을 불러옵니다. <br> <br>
CustomDataset은 내가 가지고 있는 데이터셋을 모델에게 제공할 방식을 정의한 클래스입니다. <br> <br>
CustomDataset에는 적어도<br> <br>
(1) 데이터의 목록 <br>
(2) 데이터를 저장장치에서 불러오는 방식 <br>
(3) 데이터를 전처리하는 방식 <br>
세 가지가 정의되어 있어야 하며 <br> <br>
(1) __len__ <br>
(2) __getitem__ <br>
두 가지의 메써드가 정의되어 있어야 합니다. <br> <br>

In [ ]:
dataset = CustomDataset(opt.data_root, opt.image_size, opt.is_train)
print(len(dataset))

train option에 있는 하이퍼파라미터를 CustomDataset에 입력하여 초기화합니다. <br> <br>
dataset 인스턴스의 길이 len(dataset)를 출력하면 데이터셋의 총 갯수가 출력됩니다. <br> <br>
만약 0이 출력된다면 option에 있는 데이터 경로가 잘못되었음을 의미합니다. <br> <br>

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(dataset, batch_size=opt.batch_size, shuffle=True, num_workers=opt.num_workers)
print(len(dataloader))

torch.utils.data의 DataLoader를 불러옵니다. <br> <br>
앞에서 선언한 dataset과 train option에 있는 하이퍼파라미터를 DataLaoder에 입력하여 초기화합니다. <br> <br>
dataloader 인스턴스의 길이 len(dataloader)를 출력하면 배치의 총 갯수(데이터셋의 수 / 배치사이즈)가 출력됩니다. <br> <br>
dataset의 크기가 0이라면 num_samples=0이라는 오류가 발생합니다. <br> <br> <br>

dataloader는 가지고 있는 데이터를 dataset에서 정의된 방식대로 읽고 전처리하여 <br> <br> <br>
그것을 batch_size 만큼 분할하여 계속 데이터를 리턴해주는 python iterator 입니다. (no python generator) <br> <br> <br>

In [ ]:
dataloader = iter(dataloader)
print(dataloader)

A = next(dataloader)
print(A[0].shape, A[1].shape, A[1])
B = next(dataloader)
print(B[0].shape, B[1].shape, B[1])

dataloader를 iterator로 바꾼 후 next로 계속 값을 리턴 받을 수 있는 것을 확인할 수 있습니다. <br> <br>

In [ ]:
for i, data in enumerate(dataloader):
    image, label = data
    print(i, image.size(), label.size())
    if i > 10:
        break

dataloader는 python iterator로 작동하기 때문에 for문을 통하여 데이터를 계속 리턴받을 수 있습니다. <br> <br>
for 문은 모든 데이터를 한 번 다 리턴할 때(한 epoch)까지 반복됩니다. <br> <br>
for문이 끝났을 때 shuffle=True일 경우 데이터셋의 순서를 섞어줍니다. <br> <br>
num_workers는 데이터셋을 불러오는 서브프로세스의 갯수로 >0 인 값을 입력할 경우 메인프로세스와 별개의 서브프로세스를 열어 데이터를 불러옵니다. <br> <br>
파이썬은 기본적으로 하나의 프로세스만을 사용합니다. (자세한 내용은 python GIL 검색) <br> <br>

In [ ]:
from pipeline import define_dataset

dataset, dataloader = define_dataset(opt)
print(len(dataset), len(dataloader))

dataset과 dataloader를 자동으로 불러올 수 있는 define_dataset이라는 함수가 pipeline.py 안에 정의되어 있습니다.